# 06 · Sample clustering from predicted spatial composition

Read whole-slide prediction maps, learn specimen composition embeddings, and save specimen clusters.

## 1. Read the input data

Edit the real input paths and method parameters below. Run the cells in order using the `spagc` kernel. Each notebook reads its own inputs; it does not execute another notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import umap
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/02_he_prediction").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_ID = os.environ.get("SPAGC_RUN_ID", "he_reference_v2")
OUT = ROOT / "outputs/he_prediction" / RUN_ID
DEVICE = os.environ.get(
    "SPAGC_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu"
)
FEATURE_DIR = OUT / "features"
SLIDE_ROWS = [
    ("proseg1", 984, 1784, 27761, 50000, "proseg1.tif", "proseg1.h5"),
    ("proseg2", 824, 1784, 23140, 50000, "proseg2.tif", "proseg2.h5"),
    ("proseg3", 856, 1784, 23981, 50000, "proseg3.tif", "proseg3.h5"),
    ("proseg4", 912, 1784, 25667, 50000, "proseg4.tif", "proseg4.h5"),
    ("proseg5", 896, 1784, 25108, 50000, "proseg5.tif", "proseg5.h5"),
    ("proseg6", 912, 1784, 25706, 50000, "proseg6.tif", "proseg6.h5"),
    ("proseg7", 1624, 3472, 45603, 97266, "proseg7.tif", "proseg7.h5"),
    ("proseg8", 1480, 3472, 41510, 97267, "proseg8.tif", "proseg8.h5"),
    ("proseg9", 1624, 3472, 45603, 97262, "proseg_9.tif", "proseg_9.h5"),
    ("proseg10", 1624, 3472, 45605, 97259, "proseg_10.tif", "proseg_10.h5"),
]
SLIDES = []
for (
    name,
    height,
    width,
    native_height,
    native_width,
    image_name,
    feature_name,
) in SLIDE_ROWS:
    index = name.removeprefix("proseg")
    SLIDES.append(
        {
            "name": name,
            "h5ad_path": str(
                ROOT / "data/he_aligned_h5ad" / f"proseg_{index}_aligned.h5ad"
            ),
            "h5_path": str(FEATURE_DIR / feature_name),
            "he_tif_path": str(ROOT / "data/he_slides" / image_name),
            "niche_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_niche_uint16.dat"
            ),
            "domain_label_path": str(
                ROOT
                / "data/he_export_2026_10_09/labels"
                / f"Proseg{index}"
                / "label_domain_uint16.dat"
            ),
            "label_shape": [height, width],
            "H": height,
            "W": width,
            "level0_shape": [native_height, native_width],
            "he_shape": [height * 28, width * 28],
            "region_size": 28,
        }
    )
TRAIN_SLIDES = SLIDES[:8]
VALIDATION_SLIDES = SLIDES[8:]
N_CLUSTERS = 5
PRED_DIR = OUT / "predictions"
SAMPLE_OUT = OUT / "sample_clustering"

In [ ]:
from spagc._io import require_files
from spagc.histology.metadata import prepare_slides

Read the slide records and select the real prediction directory produced by whole-slide inference.

In [ ]:
N_NICHE, N_DOMAIN, BG_LABEL = (22, 6, 0)
GRID, ADD_MASK, BLOCK_ROWS = (64, True, 2048)
EMB_DIM, OUT_DIM = (128, 64)
EPOCHS, BATCH_SIZE, LR = (200, 16, 0.001)
IN_CH = N_NICHE + N_DOMAIN + int(ADD_MASK)
device = torch.device(DEVICE)
torch.manual_seed(42)
slides = prepare_slides(TRAIN_SLIDES + VALIDATION_SLIDES, root=ROOT)
if len(slides) < N_CLUSTERS:
    raise ValueError("The requested number of clusters exceeds the sample count.")
SAMPLE_OUT.mkdir(parents=True, exist_ok=True)

## 2. Read prediction maps and build composition tensors

Read niche/domain rasters by memory mapping, check label IDs and geometry, then summarize local tissue composition on the regular lattice.

Summarize niche/domain fractions and tissue coverage on a regular slide lattice.

In [ ]:
def maps_to_grid(
    niche_map,
    domain_map,
    grid=GRID,
    n_niche=N_NICHE,
    n_domain=N_DOMAIN,
    bg_label=BG_LABEL,
    add_mask=ADD_MASK,
    block_rows=BLOCK_ROWS,
):
    """Pixel-prediction maps (ndarray or memmap) -> (C, grid, grid) composition image, streamed."""
    H, W = niche_map.shape
    ncell = grid * grid
    n_counts = np.zeros(ncell * n_niche, np.float64)  # flat (cell, class) accumulators
    d_counts = np.zeros(ncell * n_domain, np.float64)
    tissue_per_cell = np.zeros(ncell, np.float64)
    total_per_cell = np.zeros(ncell, np.float64)

    col_cell = np.arange(W) * grid // W  # (W,)

    for r0 in range(0, H, block_rows):
        r1 = min(r0 + block_rows, H)
        nb = np.asarray(niche_map[r0:r1]).ravel()  # materialize just this block
        db = np.asarray(domain_map[r0:r1]).ravel()
        row_cell = (np.arange(r0, r1) * grid // H)[:, None]
        cell = (row_cell * grid + col_cell[None, :]).ravel()

        tissue = np.ones_like(nb, bool) if bg_label is None else (nb != bg_label)
        ct = cell[tissue]

        total_per_cell += np.bincount(cell, minlength=ncell)
        tissue_per_cell += np.bincount(ct, minlength=ncell)
        n_counts += np.bincount(
            ct * n_niche + nb[tissue].clip(0, n_niche - 1), minlength=ncell * n_niche
        )
        d_counts += np.bincount(
            ct * n_domain + db[tissue].clip(0, n_domain - 1), minlength=ncell * n_domain
        )

    denom = tissue_per_cell + 1e-8
    n_frac = n_counts.reshape(ncell, n_niche).T / denom  # (n_niche,  ncell)
    d_frac = d_counts.reshape(ncell, n_domain).T / denom  # (n_domain, ncell)

    chans = [n_frac, d_frac]
    if add_mask:
        chans.append((tissue_per_cell / (total_per_cell + 1e-8))[None, :])

    return np.concatenate(chans, axis=0).reshape(-1, grid, grid).astype(np.float32)

In [ ]:
(X, sample_names) = ([], [])
for slide in slides:
    maps = {}
    for head, classes in (("niche", N_NICHE), ("domain", N_DOMAIN)):
        path = PRED_DIR / f"{slide['name']}_{head}.dat"
        require_files(path)
        shape = tuple(slide["label_shape"])
        if path.stat().st_size != int(np.prod(shape)) * 2:
            raise ValueError(f"Prediction geometry does not match manifest: {path}")
        maps[head] = np.memmap(path, dtype=np.uint16, mode="r", shape=shape)
        if maps[head].max() >= classes:
            raise ValueError(f"Prediction IDs exceed the declared vocabulary: {path}")
    X.append(maps_to_grid(maps["niche"], maps["domain"]))
    sample_names.append(slide["name"])
X = torch.tensor(np.stack(X))
assert torch.isfinite(X).all() and X.shape[1] == IN_CH
print(X.shape)

## 3. Train the composition contrastive encoder

Train the composition CNN on augmented views. Pooled 128-dimensional vectors, not projection vectors, are used for clustering.

Encode spatial composition maps with convolutions and train a separate contrastive projection head.

Contrast two views of each specimen against all other views, excluding only self-similarities.

In [ ]:
def augment(batch, chan_drop=0.2, noise_std=0.02):
    """Two-view-friendly augmentation of (B, C, H, W) composition images (label-preserving)."""
    out = batch.clone()
    for b in range(out.size(0)):  # per-sample flips + 90-deg rotations
        if torch.rand(1).item() < 0.5:
            out[b] = torch.flip(out[b], dims=[1])
        if torch.rand(1).item() < 0.5:
            out[b] = torch.flip(out[b], dims=[2])
        out[b] = torch.rot90(out[b], k=int(torch.randint(0, 4, (1,))), dims=[1, 2])
    drop = torch.rand(out.size(1), device=out.device) < chan_drop  # channel dropout
    out[:, drop] = 0.0
    out = out + noise_std * torch.randn_like(out)  # small additive noise
    return out


class CompositionCNN(nn.Module):
    """Small CNN: composition image -> sample embedding (global average pooled)."""

    def __init__(self, in_ch, emb_dim, out_dim):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, emb_dim, 3, stride=2, padding=1),
            nn.BatchNorm2d(emb_dim),
            nn.ReLU(),
        )
        self.proj_head = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, out_dim),
        )

    def forward(self, x):
        h = self.features(x)
        emb = h.mean(dim=[2, 3])  # global average pool -> (B, emb_dim)
        z = self.proj_head(emb)  # used only by the contrastive loss
        return emb, z


def nt_xent_loss(z1, z2, temperature=0.5):
    """Normalized temperature-scaled cross entropy over a batch of embeddings."""
    batch_size = z1.size(0)
    z1, z2 = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
    reps = torch.cat([z1, z2], dim=0)

    sim = F.cosine_similarity(reps.unsqueeze(1), reps.unsqueeze(0), dim=2)
    sim.masked_fill_(
        torch.eye(2 * batch_size, dtype=torch.bool, device=z1.device), -9e15
    )

    pos = torch.cat(
        [
            torch.arange(batch_size, 2 * batch_size, device=z1.device),
            torch.arange(0, batch_size, device=z1.device),
        ]
    )
    return F.cross_entropy(sim / temperature, pos)

In [ ]:
if len(X) < 2:
    raise ValueError("Contrastive training requires at least two samples.")
model = CompositionCNN(IN_CH, EMB_DIM, OUT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loader = DataLoader(
    TensorDataset(X), batch_size=min(BATCH_SIZE, len(X)), shuffle=True, drop_last=True
)
model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        (_, z1) = model(augment(batch))
        (_, z2) = model(augment(batch))
        loss = nt_xent_loss(z1, z2)
        assert torch.isfinite(loss)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach())
    if (epoch + 1) % 5 == 0:
        print({"epoch": epoch + 1, "loss": total_loss / len(loader)})

## 4. Cluster specimen vectors and save the outputs

Standardize pooled vectors and run KMeans; save embeddings, assignments, display coordinates and model weights. Arbitrary cluster indices are not automatically matched to spatial-transcriptomic ecotypes.

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = np.concatenate(
        [
            model(batch.to(device))[0].cpu().numpy()
            for (batch,) in DataLoader(TensorDataset(X), batch_size=BATCH_SIZE)
        ],
        axis=0,
    )
frame = pd.DataFrame(embeddings, index=sample_names)
frame.to_csv(SAMPLE_OUT / "he_sample_cnn_embeddings.csv")
scaled = StandardScaler().fit_transform(embeddings)
clusters = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10).fit_predict(scaled)
display_coordinates = umap.UMAP(
    n_neighbors=min(15, len(X) - 1),
    min_dist=0.1,
    n_components=2,
    random_state=42,
    n_jobs=1,
).fit_transform(scaled)
assignments = pd.DataFrame(
    {
        "sample": sample_names,
        "cluster": clusters,
        "UMAP1": display_coordinates[:, 0],
        "UMAP2": display_coordinates[:, 1],
    }
)
assignments.to_csv(SAMPLE_OUT / "he_sample_classification.csv", index=False)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_channels": IN_CH,
        "embedding_dim": EMB_DIM,
        "projection_dim": OUT_DIM,
        "niche_channels": N_NICHE,
        "domain_channels": N_DOMAIN,
    },
    SAMPLE_OUT / "composition_cnn.pth",
)
assert np.isfinite(embeddings).all() and len(assignments) == len(slides)
print(assignments)